# 💳 Modélisation Prédictive & Benchmark Multi-Modèles MLflow (Bloc 3 - RNCP 41993)

Ce notebook documente le pipeline d'entraînement supervisé conforme MLOps :

1. **Chargement intégral des données historiques :** `fraudTest.csv` (`555 719` transactions bancaires, sans échantillonnage tronqué).
2. **Feature Engineering partagé (`01_train/features.py`) :** Distance orthodromique Haversine, extraction horaire UTC (ISO & epoch ms), âge et densité urbaine.
3. **Prétraitement anti-fuite (`sklearn.pipeline.Pipeline`) :** `StandardScaler()` + classifieur encapsulés ensemble.
4. **Benchmark multi-modèles dans MLflow (`PostgreSQL` + `S3 / MinIO`) :** `LogisticRegression`, `RandomForestClassifier`, `HistGradientBoostingClassifier` (`class_weight='balanced'`).
5. **Gouvernance Model Registry :** Promotion automatique des alias `@champion` et `@challenger` dans MLflow (`models:/fraud_detection_model@champion`) et sauvegarde fallback `.joblib`.

In [ ]:
import os
import sys
import importlib
import pandas as pd
from IPython.display import display

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "01_train" else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

features_mod = importlib.import_module("01_train.features")
train_mod = importlib.import_module("01_train.train_model")

features_mod.load_project_env()
print(f"✅ Environnement prêt | MLflow URI = {os.getenv('MLFLOW_TRACKING_URI')}")

## 1. Chargement Intégral & Feature Engineering Partagé (`features.py`)

In [ ]:
csv_path = train_mod.resolve_dataset_path()
df = pd.read_csv(csv_path)
X = features_mod.prepare_features(df)
y = df["is_fraud"].astype(int)

print(f"Dataset : {csv_path}")
print(f"Transactions : {len(df):,} | Fraudes : {int(y.sum()):,} ({y.mean()*100:.2f} %)")
display(X.head())

## 2. Exécution du Benchmark Multi-Modèles & Promotion MLflow `@champion` / `@challenger`

In [ ]:
train_mod.train()

## 3. Inspection des Métadonnées & Comparatif Enregistré

In [ ]:
import json

with open(train_mod.META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

print(f"Champion sélectionné : {meta['champion_run_name']} (MLflow v{meta['mlflow_version']})")
print(f"Taille artefact .joblib : {meta['artifact_size_bytes'] / 1024:.1f} Ko")
display(pd.DataFrame([{"Modèle": b["run_name"], "Version MLflow": b["mlflow_version"], **b["metrics"]} for b in meta["benchmark"]]))